# 自动微分与训练：自己走一遍反向传播

先验证共享节点梯度，再训练一个2→4→1的tanh网络学习四个XOR点。数据是人工构造，目的在于验证训练机制，没有真实数据泛化结论。

[正文](../../01-concepts/deep-learning/README.md) · [完整标量自动微分与训练源码](../../05-code/foundations_core.py)。

In [1]:
from pathlib import Path
import sys, platform
import numpy as np
here = Path.cwd().resolve()
repo = next(p for p in [here, *here.parents] if (p / '10-Knowledge').is_dir())
sys.path.insert(0, str(repo / '10-Knowledge' / '01-ai-foundations' / '05-code'))
from foundations_core import *
np.set_printoptions(precision=6, suppress=True)
print('Python:', platform.python_version(), 'NumPy:', np.__version__)
print('Data: synthetic teaching examples; no model download or API call')

Python: 3.12.13 NumPy: 2.5.2
Data: synthetic teaching examples; no model download or API call


## 1. 一个参数被复用时，梯度必须累加

$L=w^2+w$、$w=3$时，梯度应为$2w+1=7$。如果图中只保留一个父节点贡献，容易得到错误答案。

In [2]:
w = Scalar(3.)
loss = w*w+w
loss.backward()
print('loss:',loss.data,'gradient:',w.grad)
assert loss.data == 12. and w.grad == 7.
loss.backward()
print('second backward resets gradients in this teaching engine:',w.grad)
assert w.grad == 7.

loss: 12.0 gradient: 7.0
second backward resets gradients in this teaching engine: 7.0


## 2. 检查激活函数链式法则

$L=(\tanh(wx+b)-y)^2/2$。固定$x=2,y=0.5,w=0.3,b=-0.2$，比较自动微分、手工公式和中心差分。

In [3]:
w,b = Scalar(.3),Scalar(-.2)
x,y = 2.,.5
h = (w*x+b).tanh()
error = h-y
loss = error*error*.5
loss.backward()
manual = (h.data-y)*(1-h.data*h.data)*x
f = lambda weight: .5*(np.tanh(weight*x+b.data)-y)**2
eps = 1e-6
numeric = (f(w.data+eps)-f(w.data-eps))/(2*eps)
print('autodiff:',w.grad,'manual:',manual,'finite difference:',numeric)
assert abs(w.grad-manual)<1e-12
assert abs(w.grad-numeric)<1e-8

autodiff: -0.20544064840745002 manual: -0.20544064840745002 finite difference: -0.20544064840456733


## 3. 训练XOR，检查参数更新后是否改变行为

四个输入为[-1,-1]、[-1,1]、[1,-1]、[1,1]，标签为[-1,1,1,-1]。单个线性分类边界无法分开XOR，隐藏层非线性提供表达能力。固定seed=7、1600次更新，不按结果筛选种子。

In [4]:
history,predictions = train_xor()
print('step, mean squared error')
for row in history: print(row)
print('raw predictions:',np.array(predictions))
print('signs:',np.sign(predictions).astype(int))
assert np.array_equal(np.sign(predictions),[-1,1,1,-1])
assert history[-1][1] < history[0][1]/100

step, mean squared error
(1, 1.0353889653707335)
(100, 0.0004782355018495127)
(400, 1.5528627906347272e-19)
(1600, 2.6439166276547974e-30)
raw predictions: [-1.  1.  1. -1.]
signs: [-1  1  1 -1]


## 4. 饱和激活怎样让梯度变小

只改变激活输入，直接查看tanh导数。这里展示的是局部导数，不等于整个网络所有参数的梯度。

In [5]:
for value in [0.,1.,4.,10.]:
    node = Scalar(value); out = node.tanh(); out.backward()
    print('input:',value,'tanh:',out.data,'local derivative:',node.grad)
assert Scalar(0).tanh().data == 0

input: 0.0 tanh: 0.0 local derivative: 1.0
input: 1.0 tanh: 0.7615941559557649 local derivative: 0.41997434161402614
input: 4.0 tanh: 0.999329299739067 local derivative: 0.0013409506830258655
input: 10.0 tanh: 0.9999999958776927 local derivative: 8.244614546626394e-09


## 观察与边界

三个梯度计算一致，XOR训练损失下降且四点标签正确，证明小引擎的加法、乘法、tanh、共享参数梯度与更新流程可协作。训练点与验证点相同，因此只证明拟合能力。该引擎无张量广播、GPU、Adam和checkpoint；文章中的框架循环是伪代码，不冒充已经运行的PyTorch训练。

练习：将所有权重初始化为相同值，观察隐藏单元是否能分化；比较初始化过大时的饱和梯度。